# Model Comparison - LoL Rank Prediction

Porovnání všech natrénovaných modelů a výběr nejlepšího.

In [ ]:
import os
import json
import pickle
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')

MODELS_DIR = '../models'
print('Imports OK')

In [ ]:
# Load all model metadata
models_data = []

for filename in os.listdir(MODELS_DIR):
    if filename.startswith('model_') and filename.endswith('_meta.json'):
        with open(os.path.join(MODELS_DIR, filename), 'r', encoding='utf-8') as f:
            meta = json.load(f)
            models_data.append(meta)

print(f'Found {len(models_data)} models')

In [ ]:
# Create comparison dataframe
df = pd.DataFrame(models_data)
df = df.sort_values('cv_accuracy', ascending=False).reset_index(drop=True)
df['rank'] = range(1, len(df) + 1)

# Display table
display_cols = ['rank', 'model_name', 'cv_accuracy', 'cv_std', 'f1_score', 'description']
print('\n=== MODEL COMPARISON (sorted by CV Accuracy) ===')
df[display_cols]

In [ ]:
# Bar chart - CV Accuracy
plt.figure(figsize=(12, 6))
colors = ['gold' if i == 0 else 'silver' if i == 1 else 'peru' if i == 2 else 'steelblue' 
          for i in range(len(df))]
bars = plt.barh(df['model_name'], df['cv_accuracy'], color=colors, xerr=df['cv_std'])
plt.xlabel('CV Accuracy')
plt.title('Model Comparison - CV Accuracy')
plt.xlim(0, max(df['cv_accuracy']) * 1.1)

for i, (acc, name) in enumerate(zip(df['cv_accuracy'], df['model_name'])):
    plt.text(acc + 0.005, i, f'{acc:.4f}', va='center')

plt.tight_layout()
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Scatter plot - Accuracy vs F1
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df['cv_accuracy'], df['f1_score'], s=100, c=range(len(df)), cmap='coolwarm')

for i, name in enumerate(df['model_name']):
    plt.annotate(name, (df['cv_accuracy'].iloc[i], df['f1_score'].iloc[i]), 
                fontsize=9, ha='left', va='bottom')

plt.xlabel('CV Accuracy')
plt.ylabel('F1 Score')
plt.title('Model Comparison - Accuracy vs F1 Score')
plt.tight_layout()
plt.show()

In [ ]:
# Best model info
best = df.iloc[0]
print('='*60)
print(f'BEST MODEL: {best["model_name"]}')
print('='*60)
print(f'CV Accuracy: {best["cv_accuracy"]:.4f} (+/- {best["cv_std"]:.4f})')
print(f'F1 Score: {best["f1_score"]:.4f}')
print(f'Description: {best["description"]}')

In [ ]:
# Copy best model as main model
best_short_name = best['short_name']

src_model = os.path.join(MODELS_DIR, f'model_{best_short_name}.pkl')
src_meta = os.path.join(MODELS_DIR, f'model_{best_short_name}_meta.json')
dst_model = os.path.join(MODELS_DIR, 'model.pkl')
dst_meta = os.path.join(MODELS_DIR, 'model_meta.json')

shutil.copy(src_model, dst_model)
shutil.copy(src_meta, dst_meta)

print(f'\nCopied {best["model_name"]} as main model (model.pkl)')

In [ ]:
# Save comparison results
comparison = {
    'ranking': df[['rank', 'model_name', 'short_name', 'cv_accuracy', 'cv_std', 'f1_score', 'description']].to_dict('records'),
    'best_model': best_short_name
}

with open(os.path.join(MODELS_DIR, 'models_comparison.json'), 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print('Saved models_comparison.json')

# Porovnání modelů - LoL Rank Prediction

Tento notebook načte všechny natrénované modely, porovná jejich výkonnost a vybere nejlepší model jako hlavní.

## Natrénované modely:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM
5. Gradient Boosting
6. Extra Trees
7. K-Nearest Neighbors
8. Support Vector Machine
9. Neural Network (MLP)

## 1. Import knihoven

In [ ]:
import sys
import os
import json
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cesty
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd())))
MODELS_DIR = os.path.join(BASE_DIR, 'Training', 'models')

if not os.path.exists(MODELS_DIR):
    MODELS_DIR = '../models'

print(f"Models dir: {MODELS_DIR}")

## 2. Načtení metadat všech modelů

In [ ]:
# Seznam všech modelů
MODEL_NAMES = [
    'logistic_regression',
    'random_forest',
    'xgboost',
    'lightgbm',
    'gradient_boosting',
    'extra_trees',
    'knn',
    'svm',
    'mlp'
]

# Načtení metadat
models_data = []

for name in MODEL_NAMES:
    meta_path = os.path.join(MODELS_DIR, f'model_{name}_meta.json')
    if os.path.exists(meta_path):
        with open(meta_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
            models_data.append(meta)
            print(f"✓ Načten: {meta['model_name']}")
    else:
        print(f"✗ Nenalezen: model_{name}_meta.json")

print(f"\nCelkem načteno: {len(models_data)} modelů")

## 3. Vytvoření porovnávací tabulky

In [ ]:
# Vytvoření DataFrame s výsledky
comparison_df = pd.DataFrame(models_data)

# Výběr relevantních sloupců
display_cols = ['model_name', 'cv_accuracy', 'cv_std', 'f1_score', 'description']
available_cols = [col for col in display_cols if col in comparison_df.columns]
comparison_df = comparison_df[available_cols]

# Seřazení podle CV accuracy
comparison_df = comparison_df.sort_values('cv_accuracy', ascending=False).reset_index(drop=True)
comparison_df.index = comparison_df.index + 1  # Začínáme od 1
comparison_df.index.name = 'Rank'

print("\n" + "="*80)
print("POROVNÁNÍ MODELŮ - SEŘAZENO PODLE CV ACCURACY")
print("="*80)
comparison_df

## 4. Vizualizace výsledků

In [ ]:
# Graf CV Accuracy
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart - CV Accuracy
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(comparison_df)))[::-1]
ax1 = axes[0]
bars = ax1.barh(comparison_df['model_name'], comparison_df['cv_accuracy'], color=colors, edgecolor='black')
ax1.set_xlabel('CV Accuracy')
ax1.set_title('Porovnání modelů - CV Accuracy')
ax1.set_xlim(0, 1)

# Přidání hodnot na bary
for bar, acc, std in zip(bars, comparison_df['cv_accuracy'], comparison_df['cv_std']):
    ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{acc:.4f} (±{std:.4f})', va='center', fontsize=9)

ax1.invert_yaxis()

# Bar chart - F1 Score
ax2 = axes[1]
bars2 = ax2.barh(comparison_df['model_name'], comparison_df['f1_score'], color=colors, edgecolor='black')
ax2.set_xlabel('F1 Score (weighted)')
ax2.set_title('Porovnání modelů - F1 Score')
ax2.set_xlim(0, 1)

# Přidání hodnot
for bar, f1 in zip(bars2, comparison_df['f1_score']):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{f1:.4f}', va='center', fontsize=9)

ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot - Accuracy vs F1 Score
plt.figure(figsize=(10, 8))
scatter = plt.scatter(comparison_df['cv_accuracy'], comparison_df['f1_score'],
                     s=200, c=range(len(comparison_df)), cmap='RdYlGn_r',
                     edgecolor='black', linewidth=2)

# Anotace
for i, row in comparison_df.iterrows():
    plt.annotate(row['model_name'], 
                (row['cv_accuracy'], row['f1_score']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('CV Accuracy')
plt.ylabel('F1 Score (weighted)')
plt.title('Porovnání modelů - Accuracy vs F1 Score')
plt.grid(True, alpha=0.3)

# Zvýraznění nejlepšího modelu
best_row = comparison_df.iloc[0]
plt.scatter([best_row['cv_accuracy']], [best_row['f1_score']],
           s=400, facecolors='none', edgecolors='gold', linewidth=3,
           label=f'Nejlepší: {best_row["model_name"]}')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Error bars pro CV Accuracy
plt.figure(figsize=(12, 6))

x = range(len(comparison_df))
plt.errorbar(x, comparison_df['cv_accuracy'], 
            yerr=comparison_df['cv_std'], 
            fmt='o', markersize=10, capsize=5,
            color='steelblue', ecolor='gray', elinewidth=2)

plt.xticks(x, comparison_df['model_name'], rotation=45, ha='right')
plt.ylabel('CV Accuracy')
plt.title('Porovnání modelů s intervalem spolehlivosti (±std)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Výběr nejlepšího modelu

In [ ]:
# Identifikace nejlepšího modelu
best_model_data = comparison_df.iloc[0]
best_model_name = None

# Najít short_name pro nejlepší model
for model in models_data:
    if model['model_name'] == best_model_data['model_name']:
        best_model_name = model['short_name']
        break

print("="*60)
print("★ NEJLEPŠÍ MODEL ★")
print("="*60)
print(f"Název: {best_model_data['model_name']}")
print(f"CV Accuracy: {best_model_data['cv_accuracy']:.4f} (±{best_model_data['cv_std']:.4f})")
print(f"F1 Score: {best_model_data['f1_score']:.4f}")
print(f"Popis: {best_model_data.get('description', 'N/A')}")
print("="*60)

## 6. Uložení nejlepšího modelu jako hlavního

In [ ]:
if best_model_name:
    # Načtení nejlepšího modelu
    best_model_path = os.path.join(MODELS_DIR, f'model_{best_model_name}.pkl')
    best_meta_path = os.path.join(MODELS_DIR, f'model_{best_model_name}_meta.json')
    
    if os.path.exists(best_model_path):
        # Kopírování jako hlavní model
        with open(best_model_path, 'rb') as f:
            best_model = pickle.load(f)
        
        with open(os.path.join(MODELS_DIR, 'model.pkl'), 'wb') as f:
            pickle.dump(best_model, f)
        print(f"✓ Nejlepší model uložen jako model.pkl")
        
        # Kopírování metadat
        with open(best_meta_path, 'r', encoding='utf-8') as f:
            best_meta = json.load(f)
        
        with open(os.path.join(MODELS_DIR, 'model_meta.json'), 'w', encoding='utf-8') as f:
            json.dump(best_meta, f, indent=2, ensure_ascii=False)
        print(f"✓ Metadata uložena jako model_meta.json")
    else:
        print(f"✗ Model {best_model_name} nebyl nalezen!")
else:
    print("✗ Nebyl identifikován nejlepší model!")

## 7. Uložení porovnávacích dat

In [ ]:
# Vytvoření JSON s porovnáním
comparison_result = {
    'ranking': [],
    'best_model': best_model_name,
    'timestamp': datetime.now().isoformat()
}

for i, model in enumerate(models_data):
    # Najít rank modelu
    rank = comparison_df[comparison_df['model_name'] == model['model_name']].index[0]
    
    comparison_result['ranking'].append({
        'rank': int(rank),
        'model_name': model['model_name'],
        'short_name': model['short_name'],
        'cv_accuracy': model['cv_accuracy'],
        'cv_std': model['cv_std'],
        'f1_score': model['f1_score'],
        'description': model.get('description', '')
    })

# Seřazení podle ranku
comparison_result['ranking'] = sorted(comparison_result['ranking'], key=lambda x: x['rank'])

# Uložení
with open(os.path.join(MODELS_DIR, 'models_comparison.json'), 'w', encoding='utf-8') as f:
    json.dump(comparison_result, f, indent=2, ensure_ascii=False)

print("✓ Porovnání uloženo do models_comparison.json")

## 8. Finální shrnutí

In [ ]:
print("\n" + "="*80)
print("FINÁLNÍ SHRNUTÍ")
print("="*80)
print(f"\nCelkem natrénováno: {len(models_data)} modelů")
print(f"\nPořadí modelů podle CV Accuracy:")
print("-"*60)

for i, row in comparison_df.iterrows():
    marker = "★" if i == 1 else " "
    print(f"{marker} {i}. {row['model_name']:<25} - Accuracy: {row['cv_accuracy']:.4f}")

print("-"*60)
print(f"\n★ HLAVNÍ MODEL: {best_model_data['model_name']}")
print(f"   Bude použit pro predikce v aplikaci (model.pkl)")
print("="*80)